# minGPT

## The basics of minGPT

### minGPT library
minGPT library consists of 3 files: model.py, bpe.py, and trainer.py

model.py holds the model

bpe.py is a bytes pair encoder

trainer.py trains the model.

## Download minGPT

In [1]:
# Import mingpt
!git clone https://github.com/karpathy/minGPT.git


Cloning into 'minGPT'...
remote: Enumerating objects: 489, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 489 (delta 153), reused 145 (delta 145), pack-reused 250 (from 1)
Receiving objects: 100% (489/489), 1.43 MiB | 6.70 MiB/s, done.
Resolving deltas: 100% (267/267), done.


Load everything from minGPT

In [28]:
%cd minGPT
%pip install -e .


/content/minGPT
Obtaining file:///content/minGPT
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 74.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    

## minGPT projects

projects/adder :
Trains a GPT from scratch to add numbers

project/chargpt:
Trains a GPT to be a character level language model on some input text file

demo.ipynb:
Shows a minimal usage of the GPT and Trainer in a notebook format on a simple sorting example

generate.ipynb:
Shows how one can load a pretrained GPT2 and generate text given some prompt.

Arithmetic encoder

Assymetric encoder

Calculate code word

Use transformers to calculate the probability

log of probability is code length

Calculate probabilities will change

Universal encoder:
New sample update bias

LLM difficult to verify when given bad sequence

Rare sequence:



# demo.ipynb
This is a sortlist

In [5]:
import torch
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from mingpt.utils import set_seed
set_seed(3407)

In [6]:
import pickle

class SortDataset(Dataset):
    """
    Dataset for the Sort problem. E.g. for problem length 6:
    Input: 0 0 2 1 0 1 -> Output: 0 0 0 1 1 2
    Which will feed into the transformer concatenated as:
    input:  0 0 2 1 0 1 0 0 0 1 1
    output: I I I I I 0 0 0 1 1 2
    where I is "ignore", as the transformer is reading the input sequence
    """

    def __init__(self, split, length=6, num_digits=3):
        assert split in {'train', 'test'}
        self.split = split
        self.length = length
        self.num_digits = num_digits

    def __len__(self):
        return 10000 # ...

    def get_vocab_size(self):
        return self.num_digits

    def get_block_size(self):
        # the length of the sequence that will feed into transformer,
        # containing concatenated input and the output, but -1 because
        # the transformer starts making predictions at the last input element
        return self.length * 2 - 1

    def __getitem__(self, idx):

        # use rejection sampling to generate an input example from the desired split
        while True:
            # generate some random integers
            inp = torch.randint(self.num_digits, size=(self.length,), dtype=torch.long)
            # half of the time let's try to boost the number of examples that
            # have a large number of repeats, as this is what the model seems to struggle
            # with later in training, and they are kind of rate
            if torch.rand(1).item() < 0.5:
                if inp.unique().nelement() > self.length // 2:
                    # too many unqiue digits, re-sample
                    continue
            # figure out if this generated example is train or test based on its hash
            h = hash(pickle.dumps(inp.tolist()))
            inp_split = 'test' if h % 4 == 0 else 'train' # designate 25% of examples as test
            if inp_split == self.split:
                break # ok

        # solve the task: i.e. sort
        sol = torch.sort(inp)[0]

        # concatenate the problem specification and the solution
        cat = torch.cat((inp, sol), dim=0)

        # the inputs to the transformer will be the offset sequence
        x = cat[:-1].clone()
        y = cat[1:].clone()
        # we only want to predict at output locations, mask out the loss at the input locations
        y[:self.length-1] = -1
        return x, y


In [7]:
# print an example instance of the dataset
train_dataset = SortDataset('train')
test_dataset = SortDataset('test')
x, y = train_dataset[0]
for a, b in zip(x,y):
    print(int(a),int(b))


1 -1
1 -1
0 -1
2 -1
0 -1
1 0
0 0
0 1
1 1
1 1
1 2


In [8]:
# create a GPT instance
from mingpt.model import GPT

model_config = GPT.get_default_config()
model_config.model_type = 'gpt-nano'
model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = train_dataset.get_block_size()
model = GPT(model_config)

number of parameters: 0.09M


In [9]:
# create a Trainer object
from mingpt.trainer import Trainer

train_config = Trainer.get_default_config()
train_config.learning_rate = 5e-4 # the model we're using is so small that we can go a bit faster
train_config.max_iters = 2000
train_config.num_workers = 0
trainer = Trainer(train_config, model, train_dataset)


running on device cpu


In [10]:
def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter_dt {trainer.iter_dt * 1000:.2f}ms; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")
trainer.set_callback('on_batch_end', batch_end_callback)

trainer.run()

iter_dt 0.00ms; iter 0: train loss 1.07259
iter_dt 41.98ms; iter 100: train loss 0.13719
iter_dt 39.61ms; iter 200: train loss 0.09765
iter_dt 57.76ms; iter 300: train loss 0.04261
iter_dt 39.56ms; iter 400: train loss 0.06306
iter_dt 47.29ms; iter 500: train loss 0.02889
iter_dt 45.22ms; iter 600: train loss 0.01600
iter_dt 39.05ms; iter 700: train loss 0.00540
iter_dt 39.35ms; iter 800: train loss 0.00690
iter_dt 40.36ms; iter 900: train loss 0.00720
iter_dt 40.31ms; iter 1000: train loss 0.00980
iter_dt 39.82ms; iter 1100: train loss 0.00333
iter_dt 42.86ms; iter 1200: train loss 0.00350
iter_dt 39.17ms; iter 1300: train loss 0.00119
iter_dt 53.92ms; iter 1400: train loss 0.01831
iter_dt 39.37ms; iter 1500: train loss 0.00413
iter_dt 38.15ms; iter 1600: train loss 0.00678
iter_dt 50.20ms; iter 1700: train loss 0.00108
iter_dt 41.82ms; iter 1800: train loss 0.00374
iter_dt 38.09ms; iter 1900: train loss 0.00126


In [11]:
# now let's perform some evaluation
model.eval();

In [12]:
def eval_split(trainer, split, max_batches):
    dataset = {'train':train_dataset, 'test':test_dataset}[split]
    n = train_dataset.length # naugy direct access shrug
    results = []
    mistakes_printed_already = 0
    loader = DataLoader(dataset, batch_size=100, num_workers=0, drop_last=False)
    for b, (x, y) in enumerate(loader):
        x = x.to(trainer.device)
        y = y.to(trainer.device)
        # isolate the input pattern alone
        inp = x[:, :n]
        sol = y[:, -n:]
        # let the model sample the rest of the sequence
        cat = model.generate(inp, n, do_sample=False) # using greedy argmax, not sampling
        sol_candidate = cat[:, n:] # isolate the filled in sequence
        # compare the predicted sequence to the true sequence
        correct = (sol == sol_candidate).all(1).cpu() # Software 1.0 vs. Software 2.0 fight RIGHT on this line haha
        for i in range(x.size(0)):
            results.append(int(correct[i]))
            if not correct[i] and mistakes_printed_already < 3: # only print up to 5 mistakes to get a sense
                mistakes_printed_already += 1
                print("GPT claims that %s sorted is %s but gt is %s" % (inp[i].tolist(), sol_candidate[i].tolist(), sol[i].tolist()))
        if max_batches is not None and b+1 >= max_batches:
            break
    rt = torch.tensor(results, dtype=torch.float)
    print("%s final score: %d/%d = %.2f%% correct" % (split, rt.sum(), len(results), 100*rt.mean()))
    return rt.sum()

# run a lot of examples from both train and test through the model and verify the output correctness
with torch.no_grad():
    train_score = eval_split(trainer, 'train', max_batches=50)
    test_score  = eval_split(trainer, 'test',  max_batches=50)


train final score: 5000/5000 = 100.00% correct
test final score: 5000/5000 = 100.00% correct


In [13]:
# let's run a random given sequence through the model as well
n = train_dataset.length # naugy direct access shrug
inp = torch.tensor([[0, 0, 2, 1, 0, 1]], dtype=torch.long).to(trainer.device)
assert inp[0].nelement() == n
with torch.no_grad():
    cat = model.generate(inp, n, do_sample=False)
sol = torch.sort(inp[0])[0]
sol_candidate = cat[:, n:]
print('input sequence  :', inp.tolist())
print('predicted sorted:', sol_candidate.tolist())
print('gt sort         :', sol.tolist())
print('matches         :', bool((sol == sol_candidate).all()))


input sequence  : [[0, 0, 2, 1, 0, 1]]
predicted sorted: [[0, 0, 0, 1, 1, 2]]
gt sort         : [0, 0, 0, 1, 1, 2]
matches         : True


# Example

## Instantiate GPT-2

In [59]:
from mingpt.model import GPT
model_config = GPT.get_default_config()
model_config.model_type = 'gpt2'
model_config.vocab_size = 50257 # openai's model vocabulary
model_config.block_size = 1024  # openai's model block_size (i.e. input context length)
model = GPT(model_config)

number of parameters: 124.44M


## Create dataset

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import GPT2Tokenizer

class YourDataset(Dataset):
    """ Custom dataset for training minGPT with tokenized sequences. """

    def __init__(self, tokenized_tensor, block_size=1024):
        """
        Args:
            tokenized_tensor (Tensor): A tensor containing token IDs.
            block_size (int): The maximum sequence length.
        """
        self.tokenized_text = tokenized_tensor
        self.block_size = block_size

    def __len__(self):
        return len(self.tokenized_text) - self.block_size

    def __getitem__(self, idx):
        """
        Returns:
            x (Tensor): Input sequence.
            y (Tensor): Target sequence (shifted by one token).
        """
        x = self.tokenized_text[idx:idx + self.block_size]
        y = self.tokenized_text[idx + 1:idx + self.block_size + 1]
        return x, y

# Create a tensor as a part of your dataset
tokenized_tensor = torch.randint(0, 50257, (1024,))  # Example tensor
train_dataset = YourDataset(tokenized_tensor, block_size=128)

# Now proceed with training
from mingpt.trainer import Trainer

# Trainer configuration
train_config = Trainer.get_default_config()
train_config.learning_rate = 5e-4
train_config.max_iters = 1000
train_config.batch_size = 32

# Create and start training
trainer = Trainer(train_config, model, train_dataset)
trainer.run()


running on device cpu


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


## Unit Tests


In [4]:
!python -m unittest discover tests


----------------------------------------------------------------------
Ran 0 tests in 0.000s

OK


# Create code to generate data

In [17]:
# Create a list of characters

import numpy as np
import random

def generate_random_chars(characters, probabilities, size):
    """
    Generate a list of random characters based on given probabilities.

    :param characters: List of characters to choose from.
    :param probabilities: List of probabilities corresponding to each character.
    :param size: Number of characters to generate.
    :return: List of randomly selected characters.
    """
    return np.random.choice(characters, size=size, p=probabilities).tolist()

# Example usage
characters = ['A', 'B', 'C', 'D']
probabilities = [0.1, 0.3, 0.4, 0.2]  # Probabilities must sum to 1
size = 10  # Number of characters to generate

random_chars = generate_random_chars(characters, probabilities, size)
print("Generated Random Characters:", random_chars)

Generated Random Characters: ['C', 'D', 'B', 'A', 'C', 'B', 'C', 'D', 'B', 'D']
